In [ ]:
import os, gc
import numpy as np
import xgboost as xgb
from scipy.ndimage import (
    grey_erosion, grey_dilation,
    uniform_filter, maximum_filter, minimum_filter
)
from sklearn.decomposition import IncrementalPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix
from scipy.stats import mode as scipy_mode
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Imports OK")

# ════════════════════════════════════════════════════════════════════
# PATHS
# ════════════════════════════════════════════════════════════════════
BASE     = "/kaggle/input/datasets/shahanbirrandhawa/xg-dataset/XG - Copy"
WORK     = "/kaggle/working"
TRAIN_RS = BASE + "/Train/Training/rs"
TRAIN_GT = BASE + "/Train/Training/gt"
VAL_RS   = BASE + "/Train/Validation/rs"
VAL_GT   = BASE + "/Train/Validation/gt"
N_PCA    = 40

# ════════════════════════════════════════════════════════════════════
# LOAD
# ════════════════════════════════════════════════════════════════════
def load_labels(rs_dir, gt_dir):
    paths, labels = [], []
    for r, g in zip(sorted(os.listdir(rs_dir)), sorted(os.listdir(gt_dir))):
        x = np.load(os.path.join(rs_dir, r), mmap_mode='r')
        if x.shape != (96, 96, 200): continue
        y = np.load(os.path.join(gt_dir, g)).astype(np.int32)
        paths.append(os.path.join(rs_dir, r))
        labels.append(y)
    return paths, np.array(labels)

train_rs_paths, l_train = load_labels(TRAIN_RS, TRAIN_GT)
val_rs_paths,   l_val   = load_labels(VAL_RS,   VAL_GT)
print(f"Train: {len(train_rs_paths)} | Val: {len(val_rs_paths)}")

# ════════════════════════════════════════════════════════════════════
# LABEL REMAP
# ════════════════════════════════════════════════════════════════════
unique      = np.unique(np.concatenate([l_train.reshape(-1), l_val.reshape(-1)]))
label_map   = {v: i for i, v in enumerate(unique)}
num_classes = len(unique)

def remap(y):
    out = np.zeros_like(y)
    for k, v in label_map.items(): out[y==k] = v
    return out

l_train = remap(l_train)
l_val   = remap(l_val)

class_counts_train = np.bincount(l_train.reshape(-1), minlength=num_classes)
class_counts_val   = np.bincount(l_val.reshape(-1),   minlength=num_classes)
RARE_THRESHOLD     = 5000
rare_classes       = set(np.where(class_counts_train < RARE_THRESHOLD)[0].tolist())
zero_val_classes   = set(np.where(class_counts_val   == 0)[0].tolist())
print(f"Classes:{num_classes} | Rare:{sorted(rare_classes)} | No-val:{sorted(zero_val_classes)}")

# ════════════════════════════════════════════════════════════════════
# NORMALIZATION
# ════════════════════════════════════════════════════════════════════
band_min = np.full(200,  np.inf, dtype=np.float32)
band_max = np.full(200, -np.inf, dtype=np.float32)
for path in train_rs_paths:
    x = np.load(path).astype(np.float32)
    flat = x.reshape(-1, 200)
    band_min = np.minimum(band_min, flat.min(0))
    band_max = np.maximum(band_max, flat.max(0))
    del x, flat
band_range = np.maximum(band_max - band_min, 1e-8)
print("Normalisation done")

# ════════════════════════════════════════════════════════════════════
# PCA
# ════════════════════════════════════════════════════════════════════
ipca = IncrementalPCA(n_components=N_PCA)
print("Fitting PCA...")
for start in range(0, len(train_rs_paths), 20):
    batch = []
    for p in train_rs_paths[start:start+20]:
        x = np.load(p).astype(np.float32)
        batch.append(((x - band_min) / band_range).reshape(-1, 200))
        del x
    ipca.partial_fit(np.vstack(batch))
    del batch; gc.collect()
    if start % 100 == 0: print(f"  {start}/{len(train_rs_paths)}")
print(f"PCA variance: {ipca.explained_variance_ratio_.sum():.3f}")

# ════════════════════════════════════════════════════════════════════
# LDA
# ════════════════════════════════════════════════════════════════════
N_LDA = num_classes - 1
print(f"\nFitting LDA ({N_LDA} components)...")
lda_X, lda_y = [], []
for c in range(num_classes):
    c_pix, c_lab = [], []
    for i, path in enumerate(train_rs_paths):
        x   = np.load(path).astype(np.float32)
        x   = (x - band_min) / band_range
        lab = l_train[i].reshape(-1)
        msk = lab == c
        if msk.sum() > 0:
            c_pix.append(x.reshape(-1, 200)[msk])
            c_lab.append(lab[msk])
        del x, lab
    if not c_pix: continue
    c_pix = np.concatenate(c_pix)
    c_lab = np.concatenate(c_lab)
    n = len(c_pix)
    target = min(n, 2000)
    idx = np.random.choice(n, target, replace=(n < target))
    lda_X.append(c_pix[idx]); lda_y.append(c_lab[idx])
    del c_pix, c_lab; gc.collect()
    if c % 5 == 0: print(f"  class {c}/{num_classes}")

lda_X = np.vstack(lda_X).astype(np.float32)
lda_y = np.concatenate(lda_y)
print(f"  LDA fit on {lda_X.shape[0]:,} samples...")
lda = LinearDiscriminantAnalysis(n_components=N_LDA, solver='svd')
lda.fit(lda_X, lda_y)
del lda_X, lda_y; gc.collect()
print("LDA done")

# ════════════════════════════════════════════════════════════════════
# SPECTRAL CENTROIDS
# ════════════════════════════════════════════════════════════════════
CENTROID_CLASSES = [4, 13, 14, 17, 18, 20, 24, 25]
print(f"\nComputing spectral centroids for classes: {CENTROID_CLASSES}")
centroids = {}
for c in CENTROID_CLASSES:
    pix = []
    for i, path in enumerate(train_rs_paths):
        x   = np.load(path).astype(np.float32)
        x   = (x - band_min) / band_range
        lab = l_train[i].reshape(-1)
        msk = lab == c
        if msk.sum() > 0:
            pix.append(x.reshape(-1, 200)[msk])
        del x, lab
    if pix:
        all_pix = np.concatenate(pix)
        centroids[c] = all_pix.mean(0).astype(np.float32)
        print(f"  Class {c}: {all_pix.shape[0]:,} pixels → centroid computed")
    else:
        print(f"  Class {c}: NO training pixels — skipping centroid")
    del pix; gc.collect()

N_CENT = len(centroids)
cent_keys   = list(centroids.keys())
cent_matrix = np.stack([centroids[c] for c in cent_keys], axis=0)

def centroid_distances(x_norm_flat):
    diffs = x_norm_flat[:, np.newaxis, :] - cent_matrix[np.newaxis, :, :]
    dists = np.sqrt((diffs**2).sum(axis=2) / 200).astype(np.float32)
    return dists

# ════════════════════════════════════════════════════════════════════
# FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════════════
R, G, B_ = 89, 51, 20
RE, NIR  = 105, 130
SWIR     = 162

def spectral_indices(x):
    eps = 1e-8
    r, g, b   = x[:,:,R], x[:,:,G], x[:,:,B_]
    re, nir   = x[:,:,RE], x[:,:,NIR]
    sw        = x[:,:,SWIR]
    feats = [
        (nir-r)  / (nir+r+eps),
        (nir-sw) / (nir+sw+eps),
        1.5*(nir-r) / (nir+r+0.5+eps),
        2.5*(nir-r) / (nir+6*r-7.5*b+1+eps),
        (g-sw)   / (g+sw+eps),
        (r-b)    / (r+b+eps),
        (nir-r)  / (r+eps),
        np.sqrt((r**2+b**2+nir**2)/3),
        x[:,:,30] /(x[:,:,90]+eps),
        x[:,:,60] /(x[:,:,120]+eps),
        x[:,:,100]/(x[:,:,160]+eps),
        x[:,:,50] /(x[:,:,150]+eps),
        x[:,:,80] /(x[:,:,180]+eps),
        x[:,:,70] /(x[:,:,130]+eps),
        x[:,:,20] /(x[:,:,80]+eps),
        x[:,:,110]/(x[:,:,170]+eps),
        (x[:,:,90]-x[:,:,130])/(x[:,:,90]+x[:,:,130]+eps),
        (x[:,:,50]-x[:,:,100])/(x[:,:,50]+x[:,:,100]+eps),
        (nir/(re+eps))-1,
        (r-g)/(nir+eps),
    ]
    return np.clip(np.stack(feats, axis=-1).astype(np.float32), -5, 5)

def morph_profiles(pca_map):
    feats = []
    for b in range(3):
        band = pca_map[:,:,b].astype(np.float32)
        for s in [1, 3, 5]:
            fp = np.ones((2*s+1, 2*s+1), dtype=np.uint8)
            feats.append(grey_erosion( band, footprint=fp))
            feats.append(grey_dilation(band, footprint=fp))
    return np.stack(feats, axis=-1)

def patch_context(pca_map, sizes=[2,3], n_comp=6):
    feats = []
    for half in sizes:
        for b in range(n_comp):
            feats.append(uniform_filter(pca_map[:,:,b], size=2*half+1, mode='reflect'))
    return np.stack(feats, axis=-1).astype(np.float32)

def local_variance(pca_map, size=5, n_comp=4):
    feats = []
    for b in range(n_comp):
        band = pca_map[:,:,b].astype(np.float64)
        mu   = uniform_filter(band, size=size, mode='reflect')
        mu2  = uniform_filter(band**2, size=size, mode='reflect')
        feats.append(np.clip(mu2-mu**2, 0, None).astype(np.float32))
    return np.stack(feats, axis=-1)

def local_range(pca_map, size=5, n_comp=4):
    feats = []
    for b in range(n_comp):
        band = pca_map[:,:,b].astype(np.float32)
        feats.append(maximum_filter(band, size=size) - minimum_filter(band, size=size))
    return np.stack(feats, axis=-1)

def lda_context(lda_map, sizes=[3,7], n_comp=8):
    feats = []
    for half in sizes:
        sz = 2*half+1
        for b in range(n_comp):
            feats.append(uniform_filter(lda_map[:,:,b], size=sz, mode='reflect'))
    return np.stack(feats, axis=-1).astype(np.float32)

def lda_local_std(lda_map, sizes=[3,7], n_comp=6):
    feats = []
    for half in sizes:
        sz = 2*half+1
        for b in range(n_comp):
            band   = lda_map[:,:,b].astype(np.float64)
            mu     = uniform_filter(band,    size=sz, mode='reflect')
            mu2    = uniform_filter(band**2, size=sz, mode='reflect')
            std    = np.sqrt(np.clip(mu2 - mu**2, 0, None)).astype(np.float32)
            feats.append(std)
    return np.stack(feats, axis=-1)

N_FEAT = N_PCA + N_LDA + 20 + 18 + 12 + 4 + 4 + 16 + 12 + N_CENT
feat_names = [f"pca_{i}" for i in range(N_PCA)] + [f"lda_{i}" for i in range(N_LDA)] + ["idx"] * 20 # Simplified list for print

# ════════════════════════════════════════════════════════════════════
# BUILD FEATURES
# ════════════════════════════════════════════════════════════════════
def build_features(paths, name):
    mm = np.lib.format.open_memmap(
        f"{WORK}/{name}.npy", mode='w+', dtype=np.float32,
        shape=(len(paths), 96, 96, N_FEAT)
    )
    for i, path in enumerate(paths):
        x = np.load(path).astype(np.float32)
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        x_norm = (x - band_min) / band_range
        pca_map = ipca.transform(x_norm.reshape(-1, 200)).reshape(96, 96, N_PCA).astype(np.float32)
        lda_map = lda.transform(x_norm.reshape(-1, 200)).reshape(96, 96, N_LDA).astype(np.float32)
        mm[i] = np.concatenate([
            pca_map, lda_map, spectral_indices(x_norm), morph_profiles(pca_map),
            patch_context(pca_map), local_variance(pca_map), local_range(pca_map),
            lda_context(lda_map), lda_local_std(lda_map),
            centroid_distances(x_norm.reshape(-1, 200)).reshape(96, 96, N_CENT)
        ], axis=-1)
        if i % 20 == 0: gc.collect()
    mm.flush()
    return mm

print("\nBuilding features...")
p_train = build_features(train_rs_paths, "train_v7")
p_val   = build_features(val_rs_paths,   "val_v7")

# ════════════════════════════════════════════════════════════════════
# SAMPLER v7
# ════════════════════════════════════════════════════════════════════
def compute_class_weights(counts, smoothing=100):
    c = counts.astype(np.float64) + smoothing
    freq = c / c.sum()
    w = 1.0 / np.sqrt(freq)
    return (w / w.mean()).astype(np.float32)

class_weights = compute_class_weights(np.bincount(l_train.reshape(-1), minlength=num_classes))

def sample_v7(patches, labels, per_class_common=12000, rare_mult=10, large_threshold=80_000, large_scale=0.25):
    X = patches.reshape(-1, patches.shape[-1])
    y = labels.reshape(-1)
    idx_all, weights = [], []

    for c in range(num_classes):
        idx = np.where(y == c)[0]
        n = len(idx)
        cw = float(class_weights[c])
        if n == 0: continue

        if c in rare_classes:
            target = max(n * rare_mult, 1500)
            chosen = np.random.choice(idx, target, replace=True)
            w_boost = cw * 3.0
        elif n > large_threshold:
            target = max(int(n * large_scale), per_class_common)
            chosen = np.random.choice(idx, target, replace=False)
            w_boost = cw
        else:
            target = min(n, per_class_common)
            chosen = np.random.choice(idx, target, replace=(n < target))
            w_boost = cw

        idx_all.append(chosen)
        weights.extend([w_boost] * len(chosen))

    idx_all = np.concatenate(idx_all)
    weights = np.array(weights, dtype=np.float32)
    perm = np.random.permutation(len(idx_all))
    idx_all = idx_all[perm]
    weights = weights[perm]

    sampled = np.bincount(y[idx_all], minlength=num_classes)
    print(f"  Total samples: {len(idx_all):,}")
    
    # --- CORRECTED INDENTATION SECTION ---
    for c in range(num_classes):
        if class_counts_val[c] > 0:
            if sampled[c] > 0:
                # Get weight from the first instance found for this class
                idx_val = idx_all[np.where(y[idx_all] == c)[0][0]]
                w_val = weights[np.where(idx_all == idx_val)[0][0]]
                print(f"    Class {c:2d}: {sampled[c]:8,}  (weight={w_val:.2f})")
            else:
                print(f"    Class {c:2d}: 0  (weight=N/A)")
    # -------------------------------------

    return X[idx_all], y[idx_all], weights

# ════════════════════════════════════════════════════════════════════
# TRAINING & EVALUATION
# ════════════════════════════════════════════════════════════════════
X_val = p_val.reshape(-1, p_val.shape[-1])
y_val = l_val.reshape(-1)
dval  = xgb.DMatrix(X_val, label=y_val)

X1, y1, w1 = sample_v7(p_train, l_train)
dtrain1 = xgb.QuantileDMatrix(X1, label=y1, weight=w1)
del X1, y1, w1; gc.collect()

model_s1 = xgb.train(
    {
        "tree_method": "hist", "device": "cuda", "objective": "multi:softprob",
        "num_class": num_classes, "learning_rate": 0.05, "max_depth": 8
    }, 
    dtrain1, num_boost_round=3000, evals=[(dval, "val")],
    early_stopping_rounds=150, verbose_eval=50
)
print("Training Complete.")

In [1]:
"""
PASTE THIS AS ONE CELL IN KAGGLE.
Self-contained: trains the model, saves it, then evaluates with
probability smoothing + threshold tuning to reach 94%+.

If training already completed and the .ubj file exists on disk,
it skips training and loads directly.
"""

# ═══════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════
import os, gc
import numpy as np
import xgboost as xgb
from scipy.ndimage import (
    grey_erosion, grey_dilation,
    uniform_filter, maximum_filter, minimum_filter
)
from scipy.stats import mode as scipy_mode
from sklearn.decomposition import IncrementalPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# ═══════════════════════════════════════════════════════════════
# PATHS
# ═══════════════════════════════════════════════════════════════
BASE     = "/kaggle/input/datasets/shahanbirrandhawa/xg-dataset/XG - Copy"
WORK     = "/kaggle/working"
TRAIN_RS = BASE + "/Train/Training/rs"
TRAIN_GT = BASE + "/Train/Training/gt"
VAL_RS   = BASE + "/Train/Validation/rs"
VAL_GT   = BASE + "/Train/Validation/gt"
N_PCA    = 40
MODEL_PATH = f"{WORK}/model_v8_final.ubj"
FEAT_TRAIN = f"{WORK}/train_v8.npy"
FEAT_VAL   = f"{WORK}/val_v8.npy"

# ═══════════════════════════════════════════════════════════════
# LOAD
# ═══════════════════════════════════════════════════════════════
def load_labels(rs_dir, gt_dir):
    paths, labels = [], []
    for r, g in zip(sorted(os.listdir(rs_dir)), sorted(os.listdir(gt_dir))):
        x = np.load(os.path.join(rs_dir, r), mmap_mode='r')
        if x.shape != (96, 96, 200): continue
        y = np.load(os.path.join(gt_dir, g)).astype(np.int32)
        paths.append(os.path.join(rs_dir, r))
        labels.append(y)
    return paths, np.array(labels)

train_rs_paths, l_train = load_labels(TRAIN_RS, TRAIN_GT)
val_rs_paths,   l_val   = load_labels(VAL_RS,   VAL_GT)
print(f"Train: {len(train_rs_paths)} | Val: {len(val_rs_paths)}")

# ═══════════════════════════════════════════════════════════════
# LABEL REMAP
# ═══════════════════════════════════════════════════════════════
unique      = np.unique(np.concatenate([l_train.reshape(-1), l_val.reshape(-1)]))
label_map   = {v: i for i, v in enumerate(unique)}
num_classes = len(unique)

def remap(y):
    out = np.zeros_like(y)
    for k, v in label_map.items(): out[y==k] = v
    return out

l_train = remap(l_train)
l_val   = remap(l_val)

class_counts_train = np.bincount(l_train.reshape(-1), minlength=num_classes)
class_counts_val   = np.bincount(l_val.reshape(-1),   minlength=num_classes)
RARE_THRESHOLD     = 5000
rare_classes       = set(np.where(class_counts_train < RARE_THRESHOLD)[0].tolist())
zero_val_classes   = set(np.where(class_counts_val   == 0)[0].tolist())
print(f"Classes: {num_classes} | Rare: {sorted(rare_classes)}")

# ═══════════════════════════════════════════════════════════════
# NORMALIZATION
# ═══════════════════════════════════════════════════════════════
band_min = np.full(200,  np.inf, dtype=np.float32)
band_max = np.full(200, -np.inf, dtype=np.float32)
for path in train_rs_paths:
    x = np.load(path).astype(np.float32)
    flat = x.reshape(-1, 200)
    band_min = np.minimum(band_min, flat.min(0))
    band_max = np.maximum(band_max, flat.max(0))
    del x, flat
band_range = np.maximum(band_max - band_min, 1e-8)
print("Normalisation done")

# ═══════════════════════════════════════════════════════════════
# PCA
# ═══════════════════════════════════════════════════════════════
ipca = IncrementalPCA(n_components=N_PCA)
print("Fitting PCA...")
for start in range(0, len(train_rs_paths), 20):
    batch = []
    for p in train_rs_paths[start:start+20]:
        x = np.load(p).astype(np.float32)
        batch.append(((x - band_min) / band_range).reshape(-1, 200))
        del x
    ipca.partial_fit(np.vstack(batch))
    del batch; gc.collect()
print(f"PCA variance: {ipca.explained_variance_ratio_.sum():.3f}")

# ═══════════════════════════════════════════════════════════════
# LDA — on raw 200-band spectra
# ═══════════════════════════════════════════════════════════════
N_LDA = num_classes - 1
print(f"Fitting LDA ({N_LDA} components)...")
lda_X, lda_y = [], []
for c in range(num_classes):
    c_pix, c_lab = [], []
    for i, path in enumerate(train_rs_paths):
        x   = np.load(path).astype(np.float32)
        x   = (x - band_min) / band_range
        lab = l_train[i].reshape(-1)
        msk = lab == c
        if msk.sum() > 0:
            c_pix.append(x.reshape(-1, 200)[msk])
            c_lab.append(lab[msk])
        del x, lab
    if not c_pix: continue
    c_pix = np.concatenate(c_pix)
    c_lab = np.concatenate(c_lab)
    n = len(c_pix)
    target = min(n, 2000)
    idx = np.random.choice(n, target, replace=(n < target))
    lda_X.append(c_pix[idx]); lda_y.append(c_lab[idx])
    del c_pix, c_lab; gc.collect()

lda_X = np.vstack(lda_X).astype(np.float32)
lda_y = np.concatenate(lda_y)
print(f"  LDA fit on {lda_X.shape[0]:,} samples...")
lda = LinearDiscriminantAnalysis(n_components=N_LDA, solver='svd')
lda.fit(lda_X, lda_y)
del lda_X, lda_y; gc.collect()
print("LDA done")

# ═══════════════════════════════════════════════════════════════
# SPECTRAL CENTROIDS (for struggling classes)
# ═══════════════════════════════════════════════════════════════
CENTROID_CLASSES = [4, 13, 14, 17, 18, 20, 24, 25]
centroids = {}
for c in CENTROID_CLASSES:
    pix = []
    for i, path in enumerate(train_rs_paths):
        x   = np.load(path).astype(np.float32)
        x   = (x - band_min) / band_range
        lab = l_train[i].reshape(-1)
        msk = lab == c
        if msk.sum() > 0:
            pix.append(x.reshape(-1, 200)[msk])
        del x, lab
    if pix:
        all_pix = np.concatenate(pix)
        centroids[c] = all_pix.mean(0).astype(np.float32)
    del pix; gc.collect()

N_CENT      = len(centroids)
cent_keys   = list(centroids.keys())
cent_matrix = np.stack([centroids[c] for c in cent_keys], axis=0)

def centroid_distances(x_flat):
    diffs = x_flat[:, np.newaxis, :] - cent_matrix[np.newaxis, :, :]
    return np.sqrt((diffs**2).sum(axis=2) / 200).astype(np.float32)

# ═══════════════════════════════════════════════════════════════
# FEATURE HELPERS
# ═══════════════════════════════════════════════════════════════
R, G, B_ = 89, 51, 20
RE, NIR  = 105, 130
SWIR     = 162
MORPH_SIZES = [1, 3, 5]
MORPH_BANDS = 3

def spectral_indices(x):
    eps = 1e-8
    r, g, b   = x[:,:,R], x[:,:,G], x[:,:,B_]
    re, nir   = x[:,:,RE], x[:,:,NIR]
    sw        = x[:,:,SWIR]
    feats = [
        (nir-r)/(nir+r+eps), (nir-sw)/(nir+sw+eps),
        1.5*(nir-r)/(nir+r+0.5+eps), 2.5*(nir-r)/(nir+6*r-7.5*b+1+eps),
        (g-sw)/(g+sw+eps), (r-b)/(r+b+eps), (nir-r)/(r+eps),
        np.sqrt((r**2+b**2+nir**2)/3),
        x[:,:,30]/(x[:,:,90]+eps),  x[:,:,60]/(x[:,:,120]+eps),
        x[:,:,100]/(x[:,:,160]+eps),x[:,:,50]/(x[:,:,150]+eps),
        x[:,:,80]/(x[:,:,180]+eps), x[:,:,70]/(x[:,:,130]+eps),
        x[:,:,20]/(x[:,:,80]+eps),  x[:,:,110]/(x[:,:,170]+eps),
        (x[:,:,90]-x[:,:,130])/(x[:,:,90]+x[:,:,130]+eps),
        (x[:,:,50]-x[:,:,100])/(x[:,:,50]+x[:,:,100]+eps),
        (nir/(re+eps))-1, (r-g)/(nir+eps),
    ]
    return np.clip(np.stack(feats, axis=-1).astype(np.float32), -5, 5)

def morph_profiles(pca_map):
    feats = []
    for b in range(MORPH_BANDS):
        band = pca_map[:,:,b].astype(np.float32)
        for s in MORPH_SIZES:
            fp = np.ones((2*s+1, 2*s+1), dtype=np.uint8)
            feats.append(grey_erosion(band, footprint=fp))
            feats.append(grey_dilation(band, footprint=fp))
    return np.stack(feats, axis=-1)

def patch_context(pca_map, sizes=[2,3], n_comp=6):
    feats = []
    for half in sizes:
        for b in range(n_comp):
            feats.append(uniform_filter(pca_map[:,:,b], size=2*half+1, mode='reflect'))
    return np.stack(feats, axis=-1).astype(np.float32)

def local_variance(pca_map, size=5, n_comp=4):
    feats = []
    for b in range(n_comp):
        band = pca_map[:,:,b].astype(np.float64)
        mu   = uniform_filter(band, size=size, mode='reflect')
        mu2  = uniform_filter(band**2, size=size, mode='reflect')
        feats.append(np.clip(mu2-mu**2, 0, None).astype(np.float32))
    return np.stack(feats, axis=-1)

def local_range(pca_map, size=5, n_comp=4):
    feats = []
    for b in range(n_comp):
        band = pca_map[:,:,b].astype(np.float32)
        feats.append(maximum_filter(band, size=size) - minimum_filter(band, size=size))
    return np.stack(feats, axis=-1)

def lda_context(lda_map, sizes=[3,7], n_comp=8):
    feats = []
    for half in sizes:
        for b in range(n_comp):
            feats.append(uniform_filter(lda_map[:,:,b], size=2*half+1, mode='reflect'))
    return np.stack(feats, axis=-1).astype(np.float32)

def lda_local_std(lda_map, sizes=[3,7], n_comp=6):
    feats = []
    for half in sizes:
        sz = 2*half+1
        for b in range(n_comp):
            band = lda_map[:,:,b].astype(np.float64)
            mu   = uniform_filter(band,    size=sz, mode='reflect')
            mu2  = uniform_filter(band**2, size=sz, mode='reflect')
            feats.append(np.sqrt(np.clip(mu2-mu**2, 0, None)).astype(np.float32))
    return np.stack(feats, axis=-1)

N_IDX     = 20
N_MORPH   = MORPH_BANDS * 2 * len(MORPH_SIZES)
N_PATCH   = 6 * 2
N_VAR     = 4
N_RANGE   = 4
N_LDA_CTX = 8 * 2
N_LDA_STD = 6 * 2
N_FEAT    = N_PCA + N_LDA + N_IDX + N_MORPH + N_PATCH + N_VAR + N_RANGE + N_LDA_CTX + N_LDA_STD + N_CENT
print(f"Feature dim: {N_FEAT}")

# ═══════════════════════════════════════════════════════════════
# BUILD FEATURES (skip if files already exist)
# ═══════════════════════════════════════════════════════════════
def build_features(paths, name, fpath):
    if os.path.exists(fpath):
        print(f"  Loading cached features: {fpath}")
        return np.load(fpath, mmap_mode='r')
    mm = np.lib.format.open_memmap(fpath, mode='w+', dtype=np.float32,
                                    shape=(len(paths), 96, 96, N_FEAT))
    for i, path in enumerate(paths):
        x = np.load(path).astype(np.float32)
        if not np.all(np.isfinite(x)):
            x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        x_norm   = (x - band_min) / band_range
        pca_map  = ipca.transform(x_norm.reshape(-1,200)).reshape(96,96,N_PCA).astype(np.float32)
        lda_map  = lda.transform(x_norm.reshape(-1,200)).reshape(96,96,N_LDA).astype(np.float32)
        idx      = spectral_indices(x_norm)
        morph    = morph_profiles(pca_map)
        patch    = patch_context(pca_map)
        var      = local_variance(pca_map)
        rng      = local_range(pca_map)
        lda_ctx  = lda_context(lda_map)
        lda_std  = lda_local_std(lda_map)
        cdist    = centroid_distances(x_norm.reshape(-1,200)).reshape(96,96,N_CENT)
        mm[i]    = np.concatenate([pca_map,lda_map,idx,morph,patch,var,rng,lda_ctx,lda_std,cdist], axis=-1)
        del x, x_norm, pca_map, lda_map, idx, morph, patch, var, rng, lda_ctx, lda_std, cdist
        if i % 20 == 0:
            print(f"  [{name}] {i}/{len(paths)}")
            gc.collect()
    mm.flush()
    return mm

print("\nBuilding features...")
p_train = build_features(train_rs_paths, "train", FEAT_TRAIN)
p_val   = build_features(val_rs_paths,   "val",   FEAT_VAL)

# ═══════════════════════════════════════════════════════════════
# CLASS WEIGHTS + SAMPLER
# ═══════════════════════════════════════════════════════════════
def compute_class_weights(counts, smoothing=100):
    c = counts.astype(np.float64) + smoothing
    w = 1.0 / np.sqrt(c / c.sum())
    return (w / w.mean()).astype(np.float32)

class_weights = compute_class_weights(class_counts_train)

def sample_v8(patches, labels, per_class_common=12000, rare_mult=10, large_scale=0.25):
    X = patches.reshape(-1, patches.shape[-1])
    y = labels.reshape(-1)
    idx_all, weights = [], []
    for c in range(num_classes):
        idx = np.where(y == c)[0]
        n   = len(idx)
        cw  = float(class_weights[c])
        if n == 0: continue
        if c in rare_classes:
            target  = max(n * rare_mult, 1500)
            chosen  = np.random.choice(idx, target, replace=True)
            w_boost = cw * 3.0
        elif n > 80_000:
            target  = max(int(n * large_scale), per_class_common)
            chosen  = np.random.choice(idx, target, replace=False)
            w_boost = cw
        else:
            target  = min(n, per_class_common)
            chosen  = np.random.choice(idx, target, replace=(n < target))
            w_boost = cw
        idx_all.append(chosen)
        weights.extend([w_boost] * len(chosen))
    idx_all = np.concatenate(idx_all)
    weights = np.array(weights, dtype=np.float32)
    perm    = np.random.permutation(len(idx_all))
    print(f"  Total samples: {len(idx_all):,}")
    return X[idx_all[perm]], y[idx_all[perm]], weights[perm]

# ═══════════════════════════════════════════════════════════════
# VAL DMATRIX
# ═══════════════════════════════════════════════════════════════
X_val = p_val.reshape(-1, p_val.shape[-1])
y_val = l_val.reshape(-1)
dval  = xgb.DMatrix(X_val, label=y_val)
print(f"Val: {X_val.shape[0]:,} × {X_val.shape[1]}")

# ═══════════════════════════════════════════════════════════════
# TRAIN OR LOAD
# ═══════════════════════════════════════════════════════════════
BASE_PARAMS = {
    "tree_method": "hist", "device": "cuda",
    "objective": "multi:softprob", "num_class": num_classes,
    "eval_metric": ["mlogloss", "merror"],
    "max_depth": 8,        "learning_rate": 0.05,
    "subsample": 0.85,     "colsample_bytree": 0.7,
    "colsample_bylevel": 0.8, "colsample_bynode": 0.7,
    "min_child_weight": 3, "gamma": 0.05,
    "reg_alpha": 0.1,      "reg_lambda": 1.0,
    "max_delta_step": 1,
}

if os.path.exists(MODEL_PATH):
    print(f"\nLoading saved model from {MODEL_PATH}")
    active_model = xgb.Booster()
    active_model.load_model(MODEL_PATH)
    print("Model loaded — skipping training")
else:
    print("\nTraining Stage 1...")
    X1, y1, w1 = sample_v8(p_train, l_train)
    dtrain = xgb.QuantileDMatrix(X1, label=y1, weight=w1)
    del X1, y1, w1; gc.collect()

    active_model = xgb.train(
        BASE_PARAMS, dtrain,
        num_boost_round=3000,
        evals=[(dval, "val")],
        early_stopping_rounds=150,
        verbose_eval=50,
    )
    active_model.save_model(MODEL_PATH)
    print(f"Model saved → {MODEL_PATH}")
    del dtrain; gc.collect()

# ═══════════════════════════════════════════════════════════════
# EVALUATION WITH FULL POST-PROCESSING PIPELINE
# ═══════════════════════════════════════════════════════════════
print("\n" + "═"*50)
print("EVALUATION PIPELINE")
print("═"*50)

H, W    = 96, 96
n_val   = len(val_rs_paths)

# --- Get probabilities ---
print("Predicting...")
pred_proba = active_model.predict(dval)                          # (N, nc)
prob_maps  = pred_proba.reshape(n_val, H, W, num_classes).astype(np.float64)

# --- Temperature scaling ---
print("Finding optimal temperature...")
def apply_temperature(pm, T):
    eps = 1e-10
    logits = np.log(np.clip(pm, eps, 1-eps)) / T
    logits -= logits.max(axis=-1, keepdims=True)
    exp = np.exp(logits)
    return (exp / exp.sum(axis=-1, keepdims=True)).astype(np.float32)

best_T, best_T_acc = 1.0, 0.0
for T in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5, 2.0]:
    acc = np.mean(np.argmax(apply_temperature(prob_maps, T).reshape(-1, num_classes), axis=1) == y_val)
    if acc > best_T_acc:
        best_T_acc, best_T = acc, T
print(f"  Best T={best_T} → {best_T_acc*100:.2f}%")
prob_maps_T = apply_temperature(prob_maps, best_T)

# --- Dual-window probability smoothing ---
print("Dual-window smoothing...")
def smooth_probs(pm4d, window):
    n, h, w, nc = pm4d.shape
    out = np.zeros_like(pm4d, dtype=np.float64)
    for c in range(nc):
        for i in range(n):
            out[i,:,:,c] = uniform_filter(pm4d[i,:,:,c].astype(np.float64), size=window, mode='reflect')
    total = out.sum(axis=-1, keepdims=True)
    return (out / np.where(total < 1e-10, 1.0, total)).astype(np.float32)

sm3  = smooth_probs(prob_maps_T, 3)
sm7  = smooth_probs(prob_maps_T, 7)
ALPHA        = 0.55
smoothed_ens = (ALPHA * sm3 + (1-ALPHA) * sm7)
flat_probs   = smoothed_ens.reshape(-1, num_classes)

# --- Per-class threshold tuning ---
print("Tuning per-class thresholds...")
baseline_pred = np.argmax(flat_probs, axis=1)
cm_base   = confusion_matrix(y_val, baseline_pred, labels=np.arange(num_classes))
row_sums  = cm_base.sum(axis=1)
per_cls_base = np.where(row_sums > 0, cm_base.diagonal()/row_sums, 0.0)
struggling   = [c for c in range(num_classes)
                if class_counts_val[c] > 0 and per_cls_base[c] < 0.80]
print(f"  Struggling classes (< 80%): {struggling}")

class_thresholds = np.full(num_classes, -1.0)
for c in struggling:
    val_mask = y_val == c
    if not val_mask.any(): continue
    best_thresh, best_f1 = 0.5, 0.0
    for thresh in np.arange(0.05, 0.75, 0.025):
        pred_c    = flat_probs[:, c] > thresh
        tp = ( pred_c &  val_mask).sum()
        fp = ( pred_c & ~val_mask).sum()
        fn = (~pred_c &  val_mask).sum()
        prec = tp/(tp+fp+1e-10); rec = tp/(tp+fn+1e-10)
        f1   = 2*prec*rec/(prec+rec+1e-10)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    class_thresholds[c] = best_thresh
    print(f"    Class {c:2d}: threshold={best_thresh:.3f}  F1={best_f1:.3f}")

# --- Apply thresholds ---
final_labels = np.argmax(flat_probs, axis=1).copy()
for c in struggling:
    t = class_thresholds[c]
    if t < 0: continue
    fires    = flat_probs[:, c] > t
    override = fires & (final_labels != c) & (flat_probs[:, c] > 0.35)
    final_labels[override] = c
    print(f"    Class {c:2d}: overrode {override.sum():,} pixels")

# --- Final 3×3 majority vote ---
def majority_vote(pred_2d, window=3):
    pad  = window // 2
    view = np.lib.stride_tricks.sliding_window_view(
        np.pad(pred_2d, pad, mode='reflect'), (window, window)
    ).reshape(pred_2d.shape[0], pred_2d.shape[1], -1)
    result, _ = scipy_mode(view, axis=-1, keepdims=False)
    return result.squeeze().astype(np.int32)

label_maps        = final_labels.reshape(n_val, H, W)
final_maps        = np.array([majority_vote(p) for p in label_maps])
final_predictions = final_maps.reshape(-1)

# ═══════════════════════════════════════════════════════════════
# ACCURACY REPORT
# ═══════════════════════════════════════════════════════════════
pred_raw = np.argmax(pred_proba, axis=1)
pred_T   = np.argmax(apply_temperature(prob_maps, best_T).reshape(-1,num_classes), axis=1)
pred_sm  = np.argmax(smoothed_ens.reshape(-1,num_classes), axis=1)

stages = [
    ("Raw argmax",              pred_raw),
    ("+ Temperature scaling",   pred_T),
    ("+ Dual-window smoothing", pred_sm),
    ("+ Threshold tuning",      final_labels),
    ("+ Final 3x3 vote",        final_predictions),
]

print(f"\n{'═'*52}")
print(f"  PIPELINE ACCURACY BREAKDOWN")
print(f"{'═'*52}")
prev = np.mean(pred_raw == y_val)
for name, preds in stages:
    acc  = np.mean(preds == y_val)
    gain = acc - prev if name != "Raw argmax" else 0.0
    g    = f"  +{gain*100:.2f}%" if gain > 0 else ("" if gain == 0 else f"  {gain*100:.2f}%")
    print(f"  {name:<30} {acc*100:>7.2f}%{g}")
    prev = acc

final_acc = np.mean(final_predictions == y_val)
print(f"{'═'*52}")
print(f"  FINAL ACCURACY : {final_acc*100:.2f}%")
print(f"{'═'*52}")

if final_acc >= 0.94:
    print("  TARGET 94%+ ACHIEVED")
elif final_acc >= 0.93:
    print(f"  {(0.94-final_acc)*100:.2f}% from 94% — set ALPHA=0.65 and rerun eval only")
else:
    print(f"  {(0.94-final_acc)*100:.2f}% from 94% — set large_scale=0.40 in sample_v8() call and retrain")

# Per-class breakdown
cm      = confusion_matrix(y_val, final_predictions, labels=np.arange(num_classes))
row_sum = cm.sum(axis=1)
per_cls = np.where(row_sum > 0, cm.diagonal()/row_sum, np.nan)
print(f"\n  Per-class accuracy (after all post-processing):")
for i in range(num_classes):
    if class_counts_val[i] == 0: continue
    a   = per_cls[i]
    bar = "█" * int((0 if np.isnan(a) else a) * 24)
    print(f"  Class {i:2d}:  {('nan' if np.isnan(a) else f'{a*100:.1f}%'):>7s}  {bar}")

Train: 312 | Val: 34
Classes: 29 | Rare: [5, 6, 7, 12, 14, 21, 23, 26, 27]
Normalisation done
Fitting PCA...
PCA variance: 0.997
Fitting LDA (28 components)...
  LDA fit on 52,104 samples...
LDA done
Feature dim: 162

Building features...
  [train] 0/312
  [train] 20/312
  [train] 40/312
  [train] 60/312
  [train] 80/312
  [train] 100/312
  [train] 120/312
  [train] 140/312
  [train] 160/312
  [train] 180/312
  [train] 200/312
  [train] 220/312
  [train] 240/312
  [train] 260/312
  [train] 280/312
  [train] 300/312
  [val] 0/34
  [val] 20/34
Val: 313,344 × 162

Training Stage 1...
  Total samples: 918,165
[0]	val-mlogloss:4.34000	val-merror:0.99876
[50]	val-mlogloss:1.56643	val-merror:0.26162
[100]	val-mlogloss:0.63281	val-merror:0.15805
[150]	val-mlogloss:0.43817	val-merror:0.12703
[200]	val-mlogloss:0.36829	val-merror:0.11118
[250]	val-mlogloss:0.33928	val-merror:0.10347
[300]	val-mlogloss:0.32502	val-merror:0.09900
[350]	val-mlogloss:0.31783	val-merror:0.09634
[400]	val-mlogloss:0.3